# 02 · Tokenisation Invariance (Section 4)

**Question (RQ1).** If only the token boundaries change, does a metric's agreement with human scores change?

**What we did.** Every translation was re-cut into byte-identical, more fragmented token sequences and scored again with COMET-22. Samples are grouped into eight *buckets* by fragmentation ratio (tokens divided by the tokeniser's own count), from [1.00, 1.25) to [2.75, 3.00). *Agreement* is the Spearman correlation between metric scores and human scores within one bucket. The per-segment scores of every re-cut are in `data/scores/resegmentation/`; `experiments/tokenisation/` has the code that produced them.

This notebook rebuilds the tables of Section 4 (about 7 minutes).

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# quieten library notices that do not affect any result
os.environ["TRANSFORMERS_VERBOSITY"] = "error"      # "no PyTorch found": no model is needed here
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", module="transformers")

import pandas as pd

ROOT = Path.cwd().resolve().parent          # the repository root
sys.path.insert(0, str(ROOT / "src"))
TABLES = ROOT / "results" / "tables"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

def table(name):
    """A table written by the analysis, from results/tables/."""
    return pd.read_csv(TABLES / f"{name}.csv")

In [ ]:
from tokinv import resegmentation, equal_count, layerwise
resegmentation.run()
equal_count.run()
layerwise.run()

## Table 1: agreement falls as fragmentation rises

`rho` is the Spearman correlation, across the eight buckets, between fragmentation and agreement: -1.00 means agreement falls at every step. `ci_lo` and `ci_hi` are the 95 percent segment-bootstrap interval.

In [ ]:
t1 = table("reseg_all_settings")
t1[["corpus", "labels", "setting", "rho", "ci_lo", "ci_hi", "excl0", "n_segments"]]

In [ ]:
main = t1[t1.corpus != "WMT25 CJK"]          # the twenty settings of Table 1 (CJK is in the appendix)
print(f"{(main.excl0 & (main.rho < 0)).sum()} of {len(main)} settings decline, with an interval that excludes zero")

## Table 2: agreement in each bucket, IndicMT Eval

Gujarati falls from 0.452 in the mildest bucket to 0.038 in the most fragmented one.

In [ ]:
d = table("reseg_dose_response")
d = d[(d.direction == "forward") & d.bucket.str.startswith("[")]
d.pivot(index="bucket", columns="lang", values="spearman_comet_mqm")[["guj", "tam", "mal", "mar", "hin"]]

Agreement in the most fragmented bucket of each WMT setting (Arabic falls below zero):

In [ ]:
table("reseg_wmt_top_bucket")

## Robustness

**A second vocabulary.** The same experiment with MetricX-24, re-cutting with its own mT5 vocabulary (900 segments per language). MetricX is an error score, so a falling agreement shows as a *positive* value here.

In [ ]:
table("metricx_n900_bootstrap")

**Constant length.** Only re-cuts within five percent of the tokeniser's own token count. Agreement still falls in four of five languages: where the boundaries fall matters on its own.

In [ ]:
table("equal_count_significance_corrected")

## Section 4.3: the degradation starts inside the encoder

At each encoder layer, retrieval accuracy of an English source's translation among 200 candidates (chance 0.005), for native, re-cut and romanised text.

In [ ]:
table("layerwise_peak_with_uncertainty")